In [2]:
# -*- coding: utf-8 -*-
"""Robust PPO for Gymnasium MountainCarContinuous-v0.

Solves the local optimum trap by maintaining high exploratory variance (log_std=0.0),
applying standard clipped Gaussian policy actions, proper GAE advantage bootstrapping,
and automated archive download for Google Colab.
"""

from __future__ import annotations

import csv
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal

ENV_ID = "MountainCarContinuous-v0"


@dataclass
class Config:
    env_id: str = ENV_ID
    seed: int = 42
    total_episodes: int = 1000           # Environment typically solves in < 200 episodes
    rollout_steps: int = 2048
    max_steps_per_episode: int = 999
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_ratio: float = 0.20
    learning_rate: float = 3e-4          # Unified Adam learning rate
    train_epochs: int = 10
    minibatch_size: int = 64             # Smaller minibatch for stable gradient updates
    value_coef: float = 0.5
    entropy_coef: float = 0.005          # Small entropy bonus to prevent variance collapse
    max_grad_norm: float = 0.5
    hidden_size: int = 64
    eval_episodes: int = 20
    eval_seed_offset: int = 100000
    checkpoint_every: int = 200
    print_every: int = 20
    output_dir: str = "/content/project_outputs_ppo"
    device: str = "auto"
    render_evaluation: bool = True
    auto_download_colab: bool = True     # Automatically triggers browser download in Colab


def seed_everything(seed: int) -> None:
    """Sets random seeds across all libraries for deterministic execution."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_device(name: str) -> torch.device:
    """Selects target hardware device (CUDA GPU or CPU)."""
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but unavailable.")
    return torch.device(
        "cuda" if name in {"auto", "cuda"} and torch.cuda.is_available() else "cpu"
    )


def require_gym():
    """Ensures Gymnasium and required dependencies are installed."""
    try:
        import gymnasium as gym
        return gym
    except ImportError as exc:
        raise ImportError(
            "Install with: pip install -q 'gymnasium[classic-control]' imageio imageio-ffmpeg matplotlib"
        ) from exc


def layer_init(layer: nn.Linear, std: float = np.sqrt(2.0), bias_const: float = 0.0) -> nn.Linear:
    """Orthogonal initialization for neural network layers."""
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias_const)
    return layer


class ActorCritic(nn.Module):
    """Gaussian Actor-Critic network with decoupled state-independent log_std."""
    def __init__(self, obs_dim: int, action_dim: int, hidden: int):
        super().__init__()

        # Critic Network (Value function baseline)
        self.critic = nn.Sequential(
            layer_init(nn.Linear(obs_dim, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, 1), std=1.0),
        )

        # Actor Network (Mean action prediction)
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(obs_dim, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, action_dim), std=0.01),
        )

        # Initial log_std = 0.0 (std = 1.0) ensures strong initial momentum exploration
        self.actor_logstd = nn.Parameter(torch.zeros(1, action_dim))

    def get_value(self, obs: torch.Tensor) -> torch.Tensor:
        """Computes state value V(s)."""
        return self.critic(obs).squeeze(-1)

    def get_action_and_value(
        self,
        obs: torch.Tensor,
        action: torch.Tensor | None = None,
        deterministic: bool = False,
    ):
        """Samples or evaluates actions, computing log-probabilities, entropy, and value."""
        action_mean = self.actor_mean(obs)
        action_logstd = self.actor_logstd.expand_as(action_mean)
        action_std = torch.exp(action_logstd)
        dist = Normal(action_mean, action_std)

        if deterministic:
            act = action_mean
        else:
            if action is None:
                act = dist.sample()
            else:
                act = action

        log_prob = dist.log_prob(act).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        value = self.critic(obs).squeeze(-1)

        return act, log_prob, entropy, value


def normalize_obs(obs: np.ndarray, low: np.ndarray, high: np.ndarray) -> np.ndarray:
    """Linearly maps observation bounds into symmetric [-1.0, 1.0] interval."""
    return np.clip(2.0 * (obs - low) / (high - low) - 1.0, -1.0, 1.0).astype(np.float32)


def compute_gae(
    rewards: List[float],
    values: List[float],
    dones: List[bool],
    next_value: float,
    gamma: float,
    gae_lambda: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Computes Generalized Advantage Estimation (GAE) and TD targets."""
    advantages = np.zeros(len(rewards), dtype=np.float32)
    last_gae_lam = 0.0
    num_steps = len(rewards)

    for t in reversed(range(num_steps)):
        if t == num_steps - 1:
            next_non_terminal = 1.0 - float(dones[t])
            next_val = next_value
        else:
            next_non_terminal = 1.0 - float(dones[t])
            next_val = values[t + 1]

        delta = rewards[t] + gamma * next_val * next_non_terminal - values[t]
        last_gae_lam = delta + gamma * gae_lambda * next_non_terminal * last_gae_lam
        advantages[t] = last_gae_lam

    returns = advantages + np.asarray(values, dtype=np.float32)
    return advantages, returns


def evaluate(
    model: ActorCritic,
    env_id: str,
    cfg: Config,
    device: torch.device,
    obs_low: np.ndarray,
    obs_high: np.ndarray,
):
    """Evaluates the trained policy deterministically across independent seeds."""
    gym = require_gym()
    env = gym.make(env_id, render_mode="rgb_array" if cfg.render_evaluation else None)

    returns = []
    lengths = []
    successes = []
    first_frames = []
    first_diag = {"position": [], "velocity": [], "action": []}

    model.eval()

    try:
        for ep in range(cfg.eval_episodes):
            obs, _ = env.reset(seed=cfg.seed + cfg.eval_seed_offset + ep)
            state = normalize_obs(obs, obs_low, obs_high)
            ep_ret = 0.0

            frames = []
            positions, velocities, actions = [], [], []
            success = False

            for step in range(cfg.max_steps_per_episode):
                if cfg.render_evaluation and ep == 0:
                    frame = env.render()
                    if frame is not None:
                        frames.append(np.asarray(frame))

                st = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    action, _, _, _ = model.get_action_and_value(st, deterministic=True)

                a_clipped = torch.clamp(action, -1.0, 1.0).squeeze(0).cpu().numpy()

                if ep == 0:
                    positions.append(float(obs[0]))
                    velocities.append(float(obs[1]))
                    actions.append(float(a_clipped[0]))

                next_obs, reward, terminated, truncated, _ = env.step(a_clipped)
                ep_ret += reward
                obs = next_obs
                state = normalize_obs(obs, obs_low, obs_high)

                if terminated:
                    success = True

                if terminated or truncated:
                    break

            returns.append(ep_ret)
            lengths.append(step + 1)
            successes.append(success)

            if ep == 0:
                first_frames = frames
                first_diag = {
                    "position": np.asarray(positions, np.float32),
                    "velocity": np.asarray(velocities, np.float32),
                    "action": np.asarray(actions, np.float32),
                }
    finally:
        env.close()

    return returns, lengths, successes, first_frames, first_diag


def download_colab_artifacts(output_dir: str) -> None:
    """Zips the output directory and triggers browser download in Google Colab."""
    output_path = Path(output_dir)
    zip_target = output_path.parent / output_path.name

    print(f"\n📦 Packaging artifacts from '{output_dir}' into '{zip_target}.zip'...")
    archive_path = shutil.make_archive(str(zip_target), "zip", root_dir=output_dir)
    print(f" Archive created: {archive_path}")

    try:
        from google.colab import files
        print(" Initiating automatic browser download via Google Colab...")
        files.download(archive_path)
    except ImportError:
        print(" Note: Not running inside Google Colab. Zip file is saved locally.")


def train(cfg: Config):
    """Main PPO training loop with metrics logging, evaluation, and artifact export."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    gym = require_gym()
    seed_everything(cfg.seed)
    device = choose_device(cfg.device)

    root = Path(cfg.output_dir)
    for d in ["csv", "plots", "videos", "checkpoints", "reports", "metadata", "logs"]:
        (root / d).mkdir(parents=True, exist_ok=True)

    env = gym.make(cfg.env_id)
    obs_low = np.asarray(env.observation_space.low, np.float32)
    obs_high = np.asarray(env.observation_space.high, np.float32)
    obs_dim = int(np.prod(env.observation_space.shape))
    action_dim = int(np.prod(env.action_space.shape))

    model = ActorCritic(obs_dim, action_dim, cfg.hidden_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, eps=1e-5)

    rows = []
    rolling_returns = []
    rolling_success = []
    episode_count = 0
    start_time = time.time()

    obs, _ = env.reset(seed=cfg.seed)
    state = normalize_obs(obs, obs_low, obs_high)

    episode_return = 0.0
    episode_len = 0
    episode_success = False

    while episode_count < cfg.total_episodes:
        # Rollout buffer
        b_obs, b_actions, b_logprobs, b_rewards, b_dones, b_values = [], [], [], [], [], []

        model.eval()
        for step in range(cfg.rollout_steps):
            st = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

            with torch.no_grad():
                action, logprob, _, value = model.get_action_and_value(st, deterministic=False)

            raw_action_np = action.squeeze(0).cpu().numpy()
            clipped_action = np.clip(raw_action_np, -1.0, 1.0)

            next_obs, reward, terminated, truncated, _ = env.step(clipped_action)

            b_obs.append(state.copy())
            b_actions.append(raw_action_np.copy())
            b_logprobs.append(float(logprob.item()))
            b_rewards.append(float(reward))
            b_values.append(float(value.item()))

            done = terminated or truncated
            b_dones.append(done)

            episode_return += reward
            episode_len += 1
            if terminated:
                episode_success = True

            state = normalize_obs(next_obs, obs_low, obs_high)

            if done:
                episode_count += 1
                rolling_returns.append(episode_return)
                rolling_success.append(int(episode_success))
                rows.append({
                    "episode": episode_count,
                    "episode_return": episode_return,
                    "episode_length": episode_len,
                    "success": int(episode_success),
                })

                obs, _ = env.reset(seed=cfg.seed + episode_count)
                state = normalize_obs(obs, obs_low, obs_high)
                episode_return = 0.0
                episode_len = 0
                episode_success = False

                if episode_count >= cfg.total_episodes:
                    break

        # Bootstrap next state value for terminal GAE horizon
        st = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            next_value = float(model.get_value(st).item())

        advantages, returns = compute_gae(
            b_rewards, b_values, b_dones, next_value, cfg.gamma, cfg.gae_lambda
        )

        # Convert buffers to PyTorch tensors
        obs_t = torch.as_tensor(np.asarray(b_obs, dtype=np.float32), device=device)
        actions_t = torch.as_tensor(np.asarray(b_actions, dtype=np.float32), device=device)
        logprobs_t = torch.as_tensor(np.asarray(b_logprobs, dtype=np.float32), device=device)
        advantages_t = torch.as_tensor(advantages, device=device)
        returns_t = torch.as_tensor(returns, device=device)

        # Optimize policy and value network across minibatches
        model.train()
        b_size = obs_t.size(0)
        indices = np.arange(b_size)
        loss_actor_list, loss_critic_list, entropy_list = [], [], []

        for epoch in range(cfg.train_epochs):
            np.random.shuffle(indices)
            for start in range(0, b_size, cfg.minibatch_size):
                end = start + cfg.minibatch_size
                mb_idx = indices[start:end]

                _, new_logprob, entropy, new_value = model.get_action_and_value(
                    obs_t[mb_idx], actions_t[mb_idx]
                )

                logratio = new_logprob - logprobs_t[mb_idx]
                ratio = torch.exp(logratio)

                # Minibatch advantage normalization
                mb_adv = advantages_t[mb_idx]
                mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

                # Clipped surrogate objective
                pg_loss1 = -mb_adv * ratio
                pg_loss2 = -mb_adv * torch.clamp(ratio, 1.0 - cfg.clip_ratio, 1.0 + cfg.clip_ratio)
                actor_loss = torch.max(pg_loss1, pg_loss2).mean()

                # Critic squared-error loss
                critic_loss = 0.5 * ((new_value - returns_t[mb_idx]) ** 2).mean()

                # Overall objective
                entropy_loss = entropy.mean()
                total_loss = actor_loss + cfg.value_coef * critic_loss - cfg.entropy_coef * entropy_loss

                optimizer.zero_grad()
                total_loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                optimizer.step()

                loss_actor_list.append(actor_loss.item())
                loss_critic_list.append(critic_loss.item())
                entropy_list.append(entropy_loss.item())

        # Progress reporting
        if episode_count > 0 and len(rolling_returns) >= cfg.print_every:
            rr = float(np.mean(rolling_returns[-cfg.print_every:]))
            ss = float(np.mean(rolling_success[-cfg.print_every:]))
            std_val = float(torch.exp(model.actor_logstd).mean().item())
            print(
                f"Episode {episode_count:5d}/{cfg.total_episodes} | "
                f"recent_return={rr:8.3f} | "
                f"success_rate={ss:6.2%} | "
                f"std={std_val:5.3f} | "
                f"actor_loss={np.mean(loss_actor_list):8.4f} | "
                f"critic_loss={np.mean(loss_critic_list):8.4f}"
            )

        # Checkpoint serialization
        if episode_count > 0 and episode_count % cfg.checkpoint_every == 0:
            torch.save(
                {
                    "episode": episode_count,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "config": asdict(cfg),
                },
                root / "checkpoints" / "agent_latest.pt",
            )

    env.close()

    # Save metrics to CSV
    with (root / "csv" / "episode_metrics.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    # Policy Evaluation
    print("\n--- Evaluating Trained Policy ---")
    eval_returns, eval_lengths, eval_successes, frames, diag = evaluate(
        model, cfg.env_id, cfg, device, obs_low, obs_high
    )

    with (root / "csv" / "evaluation_metrics.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["episode", "return", "length", "success"])
        writer.writeheader()
        for i in range(len(eval_returns)):
            writer.writerow({
                "episode": i + 1,
                "return": eval_returns[i],
                "length": eval_lengths[i],
                "success": int(eval_successes[i]),
            })

    np.savez(root / "logs" / "evaluation_diagnostics.npz", **diag)

    # Save evaluation rollout video (with macro_block_size=None to eliminate ffmpeg warnings)
    if cfg.render_evaluation and frames:
        try:
            import imageio.v2 as imageio
            with imageio.get_writer(
                root / "videos" / "evaluation_rollout.mp4",
                fps=30,
                codec="libx264",
                macro_block_size=None,
            ) as writer:
                for frame in frames:
                    writer.append_data(frame)
            print("Evaluation video saved successfully.")
        except Exception as exc:
            print("Video export failed:", exc)

    summary = {
        "episodes": cfg.total_episodes,
        "recent100_return": float(np.mean(rolling_returns[-100:])) if rolling_returns else None,
        "training_success_rate": float(np.mean(rolling_success[-100:])) if rolling_success else None,
        "evaluation_mean_return": float(np.mean(eval_returns)),
        "evaluation_success_rate": float(np.mean(eval_successes)),
        "evaluation_mean_length": float(np.mean(eval_lengths)),
        "evaluation_position_max": float(diag["position"].max()),
        "wall_clock_seconds": time.time() - start_time,
    }

    (root / "reports" / "final_results.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (root / "metadata" / "config.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")

    # Plot training return curve
    plt.figure(figsize=(9, 5))
    plt.plot([r["episode"] for r in rows], [r["episode_return"] for r in rows], alpha=0.3, label="Raw Return")
    if len(rows) >= 50:
        smoothed = np.convolve([r["episode_return"] for r in rows], np.ones(50)/50, mode="valid")
        plt.plot(np.arange(50, len(rows)+1), smoothed, color="red", label="Moving Avg (50 eps)")
    plt.xlabel("Episode")
    plt.ylabel("Return")
    plt.title("PPO Training Return - MountainCarContinuous-v0")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(root / "plots" / "training_return.png", dpi=180)
    plt.close()

    # Automatically compress and trigger download in Colab
    if cfg.auto_download_colab:
        download_colab_artifacts(cfg.output_dir)

    return summary


def smoke_test():
    """Runs a quick forward-pass assertion test on the environment and model."""
    gym = require_gym()
    env = gym.make(ENV_ID)
    cfg = Config(total_episodes=1)
    obs, _ = env.reset(seed=cfg.seed)
    assert obs.shape == (2,)

    model = ActorCritic(2, 1, cfg.hidden_size)
    x = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
    action, logp, ent, val = model.get_action_and_value(x)

    assert action.shape == (1, 1)
    assert torch.isfinite(action).all()
    assert torch.isfinite(logp).all()
    assert torch.isfinite(val).all()
    env.close()
    print("Smoke test: PASS ✅")


if __name__ == "__main__":
    smoke_test()

    cfg = Config(
        total_episodes=1000,
        output_dir="/content/project_outputs_ppo",
        render_evaluation=True,
        auto_download_colab=True,
    )

    summary = train(cfg)
    print("\n--- Final Summary ---")
    print(json.dumps(summary, indent=2))

Smoke test: PASS ✅
Episode    21/1000 | recent_return=   9.363 | success_rate=45.00% | std=0.696 | actor_loss= -0.0010 | critic_loss= 44.6224
Episode    26/1000 | recent_return=  43.211 | success_rate=70.00% | std=0.694 | actor_loss=  0.0000 | critic_loss= 68.3310
Episode    31/1000 | recent_return=  68.621 | success_rate=90.00% | std=0.695 | actor_loss= -0.0000 | critic_loss= 49.4597
Episode    34/1000 | recent_return=  74.998 | success_rate=95.00% | std=0.698 | actor_loss= -0.0019 | critic_loss= 35.9606
Episode    39/1000 | recent_return=  81.396 | success_rate=100.00% | std=0.696 | actor_loss= -0.0007 | critic_loss= 56.8831
Episode    44/1000 | recent_return=  82.564 | success_rate=100.00% | std=0.692 | actor_loss= -0.0022 | critic_loss= 47.0720
Episode    49/1000 | recent_return=  82.161 | success_rate=100.00% | std=0.690 | actor_loss= -0.0030 | critic_loss= 49.7520
Episode    55/1000 | recent_return=  84.904 | success_rate=100.00% | std=0.687 | actor_loss= -0.0034 | critic_loss= 5

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- Final Summary ---
{
  "episodes": 1000,
  "recent100_return": 93.29526278234047,
  "training_success_rate": 1.0,
  "evaluation_mean_return": 93.20058635692514,
  "evaluation_success_rate": 1.0,
  "evaluation_mean_length": 71.9,
  "evaluation_position_max": 0.4204024374485016,
  "wall_clock_seconds": 275.7512652873993
}
